In [1]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\santi\OneDrive\Escritorio\proyectos\produccin-de-pozos-de-gas-y-petrleo-2024.csv",
    low_memory=False
)

# 1. Eliminar columnas que no sirven para el análisis
# Por qué: vida_util, observaciones, sub_tipo_recurso e iny_co2 
# están casi vacías o son todas cero
columnas_a_eliminar = [
    'vida_util', 'observaciones', 'sub_tipo_recurso', 
    'iny_co2', 'idusuario', 'rectificado', 'habilitado',
    'idareapermisoconcesion', 'idareayacimiento', 'idempresa'
]
df = df.drop(columns=columnas_a_eliminar)
print(f"Columnas restantes: {df.shape[1]}")

# 2. Eliminar filas con nulos en columnas clave
# Por qué: sin tipo de pozo, estado o extracción no podemos clasificar el pozo
df = df.dropna(subset=['tipopozo', 'tipoestado', 'tipoextraccion', 'cuenca'])
print(f"Filas después de eliminar nulos clave: {df.shape[0]}")

# 3. Eliminar valores negativos en producción
# Por qué: físicamente imposible, son errores de carga
df = df[df['prod_pet'] >= 0]
df = df[df['prod_gas'] >= 0]
df = df[df['prod_agua'] >= 0]
print(f"Filas después de eliminar negativos: {df.shape[0]}")

# 4. Convertir fechas a formato datetime
# Por qué: para poder operar con ellas si las necesitamos
df['fechaingreso'] = pd.to_datetime(df['fechaingreso'], errors='coerce')
df['fecha_data'] = pd.to_datetime(df['fecha_data'], errors='coerce')

# 5. Filtrar solo pozos productivos
# Por qué: para el análisis de producción no nos interesan 
# pozos abandonados o de monitoreo
estados_activos = [
    'Extracción Efectiva',
    'Otras Situación Activo',
    'Mantenimiento de Presión',
    'Parado Transitoriamente',
    'En Inyección Efectiva'
]
df_produccion = df[df['tipoestado'].isin(estados_activos)]
print(f"Filas con pozos activos: {df_produccion.shape[0]}")

# 6. Guardar el dataset limpio
df_produccion.to_csv(
    r"C:\Users\santi\OneDrive\Escritorio\proyectos\datos_limpios.csv",
    index=False
)
print("Dataset limpio guardado correctamente")

Columnas restantes: 29
Filas después de eliminar nulos clave: 983477
Filas después de eliminar negativos: 983476
Filas con pozos activos: 463039
Dataset limpio guardado correctamente
